### Election Data Usecase
[Source](https://github.com/azurede007/data-repo/blob/main/election_data_2014.csv) <br/>
File contains below fields
- state: Name of the state where the constituency is located.
- constituency:  Name of the electoral constituency.
- candidate_name:  Name of the contesting candidate.
- sex:  Gender of the candidate (M/F).
- age:  Age of the candidate in years.
- category:  Candidate’s reservation category (e.g., ST, SC, General).
- partyname:  Name of the political party the candidate represents.
- partysymbol:  Symbol associated with the candidate’s political party.
- general:  Number of votes cast via general voting method.
- postal:  Number of votes cast via postal ballots.
- total:  Total votes received by the candidate (general + postal).
- pct_of_total_votes:  Candidate’s votes as a percentage of total registered voters.
- pct_of_polled_votes:  Candidate’s votes as a percentage of total votes actually cast.
- totalvoters:  Total number of registered voters in the constituency.

### Basic Usecases
- Read the election CSV file into a PySpark DataFrame
- Infer or define a custom schema for the dataset
- Display schema and preview data with printSchema() and show()
- Select specific columns (select) and rename them (withColumnRenamed)
- Filter rows based on conditions (e.g., candidates above a certain age)
- Drop unnecessary columns using drop()
- Handle missing values using na.fill() or na.drop()
- Cast columns to correct data types (e.g., age → Integer)
- Add a new column (e.g., total votes = general + postal)
- Sort candidates by total votes in descending order
- Count total number of candidates, parties, or constituencies
- Group by state or partyname and count candidates
- Show distinct party names, categories, or constituencies
- Create a temporary view and query the data using Spark SQL

### Advanced UseCase
- Add derived columns like vote_margin, turnout_percentage, and vote_share
- Validate data consistency (e.g., check if total = general + postal)
- Clean and standardize string columns (e.g., trim spaces in constituency)
- Calculate total votes by partyname per state
- Compute average age of candidates per party or constituency
- Calculate voter turnout percentage per constituency
- Find party-wise total votes and percentage share
- Identify winner per constituency
- Find top 3 candidates per constituency and calculate margin of victory
- Rank parties by total votes per state
- Create temp views and run SQL queries to analyze party performance
- Build an end-to-end ETL pipeline (ingest → transform → aggregate → load)
- Write aggregated results into Delta tables

In [0]:
from pyspark.sql.functions import col,max,count,row_number,sum,round
from pyspark.sql.window import Window
from pyspark.sql.types import *
count_sch= StructType([StructField('state', StringType(), True), StructField('constituency', StringType(), True), StructField('candidate_name', StringType(), True), StructField('sex', StringType(), True), StructField('age', IntegerType(), True), StructField('category', StringType(), True), StructField('partyname', StringType(), True), StructField('partysymbol', StringType(), True), StructField('general', IntegerType(), True), StructField('postal', IntegerType(), True), StructField('total', FloatType(), True), StructField('pct_of_total_votes', FloatType(), True), StructField('pct_of_polled_votes', FloatType(), True), StructField('totalvoters', IntegerType(), True)])

df=spark.read.format("csv").option("header",True).schema(count_sch).load("/Volumes/databricks_practice/inputdb/election_data/election_data_2014.csv")
#print(df.schema)
df.printSchema()

#Select specific columns (select) and rename them (withColumnRenamed)
df.select("state","constituency","candidate_name","sex","age","category","partyname","partysymbol","general","postal","total","pct_of_total_votes","pct_of_polled_votes","totalvoters").show(5)

df1=df.withColumnRenamed("candidate_name","candidates_name")
df1.printSchema()
#sql way
df2=df.selectExpr("state","constituency","candidate_name as candidates_name","sex","age","category","partyname","partysymbol","general","postal","total","pct_of_total_votes","pct_of_polled_votes","totalvoters")
df2.printSchema()

#Filter rows based on conditions (e.g., candidates above a certain age)
df.filter("age>40").show(3)
#where
df.where("age>40").show(3)




In [0]:
#Drop unnecessary columns using drop()
df.drop("general","postal").show(3)
#Handle missing values using na.fill() or na.drop()
df.na.drop().show(2)
df.na.fill({"age":0}).show(3)

In [0]:
#Cast columns to correct data types (e.g., age → Integer)
df.withColumn("cast_age",col("age").cast(IntegerType())).show(3)

In [0]:
#Add a new column (e.g., total votes = general + postal)
df3=df.withColumn("total_votes",col("general")+col("postal"))
#Sort candidates by total votes in descending order
df3.orderBy("totalvoters").show(4)


In [0]:
#Count total number of candidates, parties, or constituencies
from pyspark.sql.functions import countDistinct
df4=df3.select("candidate_name","partyname","constituency").distinct()
df4.agg(countDistinct("candidate_name").alias("count_candidate"),countDistinct("partyname").alias("count_partyname"),countDistinct("constituency").alias("count_constituency")).show()

In [0]:
#Group by state or partyname and count candidates
df3.groupBy("state").agg(count("candidate_name").alias("total_candidates")).show(3)
#Show distinct party names, categories, or constituencies
df3.select("partyname","category").distinct().show()
#Create a temporary view and query the data using Spark SQL
df3.createOrReplaceTempView("election_v")
spark.sql("select * from election_v limit 1").show()

In [0]:
#Advance:
#Add derived columns like vote_margin, turnout_percentage, and vote_share
from pyspark.sql.functions import col, dense_rank, max as spark_max,trim
from pyspark.sql.window import Window

df1.createOrReplaceTempView("election_view")

spark.sql("""select * from election_view""").show(3)
window_spec=Window.partitionBy("state","constituency").orderBy(col("total").desc())
ranked_df=df1.withColumn("rank_id",dense_rank().over(window_spec))
winner_df = ranked_df.filter(col("rank_id") == 1).withColumnRenamed("total", "winner_votes")
runnerup_df = ranked_df.filter(col("rank_id") == 2).withColumnRenamed("total", "runnerup_votes")

vote_margin_df = winner_df.join(
    runnerup_df,
    on=["state", "constituency"],
    how="inner"
).withColumn(
    "vote_margin",
    col("winner_votes") - col("runnerup_votes")
)

vote_margin_df.show(2)
#Validate data consistency (e.g., check if total = general + postal)
df1.filter(col("general")+col("postal")!=col("total")).show(2)

#Clean and standardize string columns (e.g., trim spaces in constituency
df1.withColumn("trim_constituency",trim(col("constituency"))).show(5)
from pyspark.sql.functions import sum, col,avg,round
#Calculate total votes by partyname per state
df1.groupBy(col("state"), col("partyname")).agg(sum("total").alias("total_votes")).show(2)
#Compute average age of candidates per party or constituency
df1.groupBy( col("partyname")).agg(round(avg("age"),2).alias("avg_age")).show(2)


In [0]:
#Calculate voter turnout percentage per constituency
vote_per=df1.agg(sum(col("total"))/max(col("totalvoters"))*100).collect()[0][0]
print(vote_per)

#Find party-wise total votes and percentage share
df1.groupBy(col("partyname"))

In [0]:
from pyspark.sql.functions import sum
df1.groupBy(col("partyname")).agg(sum(col("pct_of_polled_votes")).alias("party_wise_vote_perct")).show()

In [0]:

#Identify winner per constituency
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank
df1.groupBy(col("constituency"))
window_spec=Window.partitionBy(col("constituency")).orderBy(col("total").desc())
df1.withColumn("winner_rank",rank().over(window_spec)).filter(col("winner_rank")==1).show()


In [0]:
#Find top 3 candidates per constituency and calculate margin of victory
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank
window_specs=Window.partitionBy(col("constituency")).orderBy(col("total").desc())
df.withColumn("rank", rank().over(window_specs)).filter(col("rank") <= 3).show(2)

In [0]:
#Rank parties by total votes per state
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, rank

# Step 1: Calculate total votes per party per state
party_votes_df = df.groupBy("state", "partyname") \
                   .agg(sum("total").alias("total_votes"))

# Step 2: Define window for ranking within each state
win = Window.partitionBy("state").orderBy(col("total_votes").desc())

# Step 3: Rank parties by total votes within each state
ranked_df = party_votes_df.withColumn("rank", rank().over(win))

ranked_df.show(truncate=False)


In [0]:
#Write aggregated results into Delta tables
ot_path="dbfs:/Volumes/databricks_practice/outputdb/election_results"
ranked_df.write.format("delta").mode("overwrite").save(ot_path)
ranked_df.write.saveAsTable("databricks_practice.outputdb.tbl_agg_elec")


In [0]:
%sql
select * from databricks_practice.outputdb.tbl_agg_elec